# Direct In-Notebook GGUF Image-to-Image Studio
## With Character Reference Analysis

**Run Cell 1-4 from Setup notebook first!**

This notebook analyzes uploaded images for character features and generates variations using Image-to-Image.

---
## Cell 6: Image Upload & Character Analysis

In [ ]:
# Cell 6: Image Upload & Character Reference Analyzer
import os
import sys
from pathlib import Path
from PIL import Image
import numpy as np

%store -r MASTER_DIR
%store -r IS_COLAB

REFERENCE_DIR = f'{MASTER_DIR}/reference_images'
os.makedirs(REFERENCE_DIR, exist_ok=True)

print("[1/3] Setting up image upload handler...")

if IS_COLAB:
    print("✓ Colab detected - using file upload")
    from google.colab import files
    
    def upload_reference_image():
        print("\nClick 'Choose Files' to upload reference image(s)...")
        uploaded = files.upload()
        
        uploaded_paths = []
        for filename, data in uploaded.items():
            filepath = f"{REFERENCE_DIR}/{filename}"
            with open(filepath, 'wb') as f:
                f.write(data)
            uploaded_paths.append(filepath)
            print(f"  ✓ Uploaded: {filename}")
        
        return uploaded_paths
else:
    print("✓ Using local file system")
    
    def upload_reference_image():
        print("\nPlace reference images in:")
        print(f"  {REFERENCE_DIR}/")
        print("\nSupported formats: .jpg, .png, .webp")
        
        # Auto-detect existing images
        valid_extensions = {'.jpg', '.jpeg', '.png', '.webp'}
        images = [f for f in os.listdir(REFERENCE_DIR) 
                  if Path(f).suffix.lower() in valid_extensions]
        
        if images:
            return [f"{REFERENCE_DIR}/{img}" for img in images]
        return []

print("[2/3] Initializing character analysis pipeline...")

try:
    import torch
    from transformers import CLIPProcessor, CLIPModel
    print("✓ Vision model support available")
except ImportError:
    print("⚠ Installing vision models...")
    import subprocess
    subprocess.run(['pip', 'install', 'transformers', 'pillow', 'numpy'], capture_output=True)
    import torch
    from transformers import CLIPProcessor, CLIPModel
    print("✓ Vision models installed")

print("[3/3] Loading CLIP for image analysis...")

# Character analysis prompts
CHARACTER_ANALYSIS_PROMPTS = {
    'body_type': [
        'slim and lean figure',
        'curvy and voluptuous figure',
        'athletic and muscular build',
        'average build',
        'petite frame',
        'tall and statuesque'
    ],
    'hair': [
        'long flowing hair',
        'short hair',
        'curly hair',
        'straight hair',
        'wavy hair',
        'dark hair',
        'blonde hair',
        'red hair'
    ],
    'facial_features': [
        'sharp facial features',
        'soft facial features',
        'high cheekbones',
        'delicate features',
        'strong jawline'
    ],
    'skin_tone': [
        'fair skin',
        'olive skin',
        'tan skin',
        'dark skin'
    ],
    'style': [
        'professional clothing',
        'casual clothing',
        'formal attire',
        'minimal clothing',
        'artistic pose'
    ]
}

print("\n" + "="*70)
print("IMAGE-TO-IMAGE CHARACTER REFERENCE ANALYZER")
print("="*70)
print("\nCapabilities:")
print("  • Upload reference images")
print("  • Analyze body type, hair, features, skin tone")
print("  • Extract detailed character descriptions")
print("  • Generate variations using Image-to-Image")

print("\n✓ Image analysis ready!")

%store REFERENCE_DIR
%store CHARACTER_ANALYSIS_PROMPTS
%store upload_reference_image

---
## Cell 6b: Upload Reference Image

In [ ]:
# Cell 6b: Upload Reference Image
%store -r upload_reference_image
%store -r REFERENCE_DIR
%store -r IS_COLAB

print("Uploading reference image for character analysis...\n")
uploaded_images = upload_reference_image()

if uploaded_images:
    print(f"\n✓ {len(uploaded_images)} reference image(s) ready for analysis")
    
    # Display uploaded images
    for img_path in uploaded_images:
        img = Image.open(img_path)
        print(f"\nImage: {os.path.basename(img_path)} | Size: {img.size}")
        if IS_COLAB:
            display(img.resize((300, 400)))
    
    %store uploaded_images
else:
    print("⚠ No images uploaded yet")
    if IS_COLAB:
        print("Run this cell and click 'Choose Files'")
    else:
        print(f"Add images to: {REFERENCE_DIR}/")

---
## Cell 6c: Analyze Character Features

In [ ]:
# Cell 6c: Detailed Character Feature Analysis
from PIL import Image
import torch
from transformers import CLIPProcessor, CLIPModel

%store -r uploaded_images
%store -r CHARACTER_ANALYSIS_PROMPTS

if not uploaded_images:
    print("❌ No reference images. Run Cell 6b first.")
else:
    print("[1/3] Loading CLIP vision model...")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    
    print("[2/3] Analyzing character reference...")
    
    analysis_results = {}
    
    for img_path in uploaded_images:
        print(f"\n  Analyzing: {os.path.basename(img_path)}")
        
        image = Image.open(img_path).convert('RGB')
        results = {}
        
        # Analyze each category
        for category, prompts in CHARACTER_ANALYSIS_PROMPTS.items():
            full_prompts = [f"A person with {p}" for p in prompts]
            
            with torch.no_grad():
                inputs = processor(text=full_prompts, images=[image]*len(prompts), 
                                 return_tensors="pt", padding=True).to(device)
                outputs = model(**inputs)
                logits_per_image = outputs.logits_per_image
                probs = logits_per_image.softmax(dim=1)[0].cpu().numpy()
            
            # Find best match
            best_idx = probs.argmax()
            results[category] = {
                'detected': prompts[best_idx],
                'confidence': float(probs[best_idx])
            }
        
        analysis_results[img_path] = results
    
    print("\n" + "="*70)
    print("CHARACTER ANALYSIS RESULTS")
    print("="*70)
    
    for img_path, analysis in analysis_results.items():
        print(f"\n📸 {os.path.basename(img_path)}")
        for category, data in analysis.items():
            print(f"  {category.upper():20} : {data['detected']:25} [{data['confidence']:.1%}]")
    
    # Build character prompt
    print("\n" + "="*70)
    print("GENERATED CHARACTER PROMPT")
    print("="*70)
    
    character_prompt = ", ".join([v['detected'] for v in list(analysis_results.values())[0].values()])
    print(f"\n{character_prompt}")
    
    %store character_prompt
    %store analysis_results

---
## Cell 6d: Image-to-Image Generation with Reference

In [ ]:
# Cell 6d: Image-to-Image Generation Preserving Character Features
import torch
from diffusers import StableDiffusionImg2ImgPipeline, DPMSolverMultistepScheduler
from PIL import Image
import os
from datetime import datetime

%store -r MASTER_DIR
%store -r MODELS_DIR
%store -r uploaded_images
%store -r character_prompt
%store -r IS_COLAB

OUTPUT_DIR = f'{MASTER_DIR}/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================
# EDIT THESE PARAMETERS
# ============================================
selected_reference = uploaded_images[0] if uploaded_images else None
additional_prompt = "wearing a red dress, in a garden setting, professional photography, 8k"
negative_prompt = "blurry, distorted, low quality, bad anatomy"
strength = 0.75  # 0.0-1.0, how much to change the image (0=no change, 1=complete regeneration)
guidance_scale = 7.5
num_inference_steps = 25
num_images = 1

if not selected_reference or not uploaded_images:
    print("❌ No reference image. Run Cell 6b to upload first.")
else:
    print(f"\n{'='*70}")
    print(f"IMAGE-TO-IMAGE GENERATION WITH CHARACTER REFERENCE")
    print(f"{'='*70}")
    print(f"Reference Image: {os.path.basename(selected_reference)}")
    print(f"\nCharacter Base: {character_prompt[:100]}...")
    print(f"Additional Context: {additional_prompt}")
    print(f"\nStrength: {strength} (0.5=subtle, 0.75=moderate, 1.0=maximum)")
    print(f"Generating {num_images} variation(s)...\n")
    
    try:
        device = "cuda" if torch.cuda.is_available() else "cpu"
        torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
        
        print("[1/4] Loading Image-to-Image model...")
        pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
            "runwayml/stable-diffusion-v1-5",
            torch_dtype=torch_dtype,
            safety_checker=None
        )
        pipe = pipe.to(device)
        pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
        
        print("[2/4] Loading reference image...")
        init_image = Image.open(selected_reference).convert('RGB')
        init_image = init_image.resize((512, 512))
        
        print("[3/4] Combining prompts...")
        full_prompt = f"{character_prompt}, {additional_prompt}"
        print(f"  Combined prompt: {full_prompt[:80]}...")
        
        print("[4/4] Running Image-to-Image inference...")
        
        with torch.no_grad():
            images = pipe(
                prompt=[full_prompt] * num_images,
                negative_prompt=[negative_prompt] * num_images,
                image=init_image,
                strength=strength,
                guidance_scale=guidance_scale,
                num_inference_steps=num_inference_steps
            ).images
        
        print(f"\n✓ Generation complete! Saving {len(images)} variation(s)...")
        
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        
        for idx, img in enumerate(images, 1):
            output_path = f"{OUTPUT_DIR}/img2img_{timestamp}_{idx}.png"
            img.save(output_path)
            print(f"  Saved: {output_path}")
            
            if IS_COLAB:
                display(img)
        
        # Cleanup
        del pipe
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
        
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        import traceback
        traceback.print_exc()

---
## Cell 6e: Batch Generate Multiple Variations

In [ ]:
# Cell 6e: Generate Multiple Variations with Different Styles
import torch
from diffusers import StableDiffusionImg2ImgPipeline
from PIL import Image
import os
from datetime import datetime

%store -r MASTER_DIR
%store -r uploaded_images
%store -r character_prompt
%store -r IS_COLAB

OUTPUT_DIR = f'{MASTER_DIR}/outputs'

# ============================================
# BATCH GENERATION - MULTIPLE STYLES
# ============================================
batch_styles = [
    "casual setting, relaxed pose",
    "professional setting, studio lighting",
    "outdoor setting, natural lighting",
    "artistic pose, dramatic lighting",
]

if not uploaded_images:
    print("❌ No reference image uploaded.")
else:
    print(f"\n{'='*70}")
    print(f"BATCH IMAGE-TO-IMAGE GENERATION")
    print(f"{'='*70}")
    print(f"Character: {character_prompt[:80]}...")
    print(f"\nGenerating {len(batch_styles)} variations...\n")
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    
    try:
        pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
            "runwayml/stable-diffusion-v1-5",
            torch_dtype=torch_dtype,
            safety_checker=None
        ).to(device)
        
        init_image = Image.open(uploaded_images[0]).convert('RGB').resize((512, 512))
        
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        all_images = []
        
        for style_idx, style in enumerate(batch_styles, 1):
            full_prompt = f"{character_prompt}, {style}"
            
            print(f"[{style_idx}/{len(batch_styles)}] {style}")
            
            with torch.no_grad():
                result = pipe(
                    prompt=full_prompt,
                    negative_prompt="blurry, distorted",
                    image=init_image,
                    strength=0.75,
                    guidance_scale=7.5,
                    num_inference_steps=20
                )
                img = result.images[0]
                all_images.append(img)
                
                output_path = f"{OUTPUT_DIR}/batch_{timestamp}_{style_idx}.png"
                img.save(output_path)
                print(f"  ✓ Saved")
                
                if IS_COLAB:
                    display(img)
        
        print(f"\n✓ Batch generation complete!")
        print(f"✓ Saved {len(all_images)} images to {OUTPUT_DIR}")
        
        del pipe
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
        
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        import traceback
        traceback.print_exc()